# 0.21 — Quantum keyword **time series** (validation lexicon)

When does the **quantum-computing** vocabulary show up in Bloomberg headlines? This is the
validation layer (annotation only — never an input to detection), the quantum analogue of `0.11`.

**Investability markers:** BQTUM index **2015-12-18** · QTUM ETF **2018-09-04** · **WQTM**
(first *pure* quantum ETF, our benchmark) **2025-10-09**. Note the index/ETF (2015/2018)
*predate* the pure-play news wave (2021–22) — quantum is a *product-led* theme.

Window here: **2013–2023** (2013–18 back-scan + 2019–20 baseline + 2021–23 corpus).

In [5]:
import re
from pathlib import Path
import numpy as np, pandas as pd
import plotly.graph_objects as go
_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = _ROOT / "notebooks" / "output"
BQTUM_INDEX = pd.Timestamp("2015-12-18"); QTUM = pd.Timestamp("2018-09-04"); WQTM = pd.Timestamp("2025-10-09")

# quantum validation lexicon (annotation only)
KEY = {  # term -> regex (d-wave needs word boundary to avoid 'secon-d-wave')
    "ionq": r"\bionq\b", "rigetti": r"rigetti", "d-wave": r"\bd-wave\b|\bdwave\b",
    "quantum computing": r"quantum comput", "qubit": r"\bqubit", "quantinuum": r"quantinuum",
}
QUANTUM = re.compile("|".join(KEY.values()), re.I)
MILESTONES = [("BQTUM index","2015-12-18"), ("QTUM ETF","2018-09-04"), ("IonQ SPAC","2021-02-24"),
              ("IonQ IPO","2021-10-01"), ("Rigetti public","2022-03-02"), ("D-Wave public","2022-08-08")]
print("lexicon terms:", len(KEY))

lexicon terms: 6


In [6]:
# load back-scan 2013-2018 (if present) + 2019-2020 baseline + 2021-2023 corpus
parts = []
_back = OUTPUT_DIR/"quantum_headlines_2013_2018.parquet"
if _back.exists():
    parts.append(pd.read_parquet(_back)[["Headline","date"]])                                    # 2013-2018
parts.append(pd.read_parquet(OUTPUT_DIR/"baseline_2019_2020_terms.parquet", columns=["Headline","date"]))  # 2019-2020
parts.append(pd.read_parquet(OUTPUT_DIR/"genai_full_meta.parquet", columns=["Headline","date"]))           # 2021-2023
df = pd.concat(parts, ignore_index=True)
df["date"] = pd.to_datetime(df["date"])
df = df.drop_duplicates("Headline")
print(f"{len(df):,} distinct headlines, {df.date.min().date()} -> {df.date.max().date()}")

hits = df[df.Headline.str.contains(QUANTUM, na=False)].copy()    # one pass -> small subset
hits["month"] = hits.date.dt.to_period("M").dt.to_timestamp()
for name, pat in KEY.items():
    hits[name] = hits.Headline.str.contains(pat, case=False, regex=True)
hits["total"] = True
ts = hits.groupby("month")[["total"]+list(KEY)].sum().sort_index()
ts = ts.reindex(pd.date_range(ts.index.min(), ts.index.max(), freq="MS"), fill_value=0)
print(f"\n{len(hits):,} quantum-lexicon headlines · monthly time series ({len(ts)} months)")
print("\nmonthly counts (total + key terms):")
print(ts.astype(int).to_string())

6,895,542 distinct headlines, 2013-05-16 -> 2023-12-30

385 quantum-lexicon headlines · monthly time series (128 months)

monthly counts (total + key terms):
            total  ionq  rigetti  d-wave  quantum computing  qubit  quantinuum
2013-05-01      2     0        0       2                  1      0           0
2013-06-01      0     0        0       0                  0      0           0
2013-07-01      1     0        0       0                  1      0           0
2013-08-01      0     0        0       0                  0      0           0
2013-09-01      0     0        0       0                  0      0           0
2013-10-01      1     0        0       1                  0      0           0
2013-11-01      0     0        0       0                  0      0           0
2013-12-01      0     0        0       0                  0      0           0
2014-01-01      0     0        0       0                  0      0           0
2014-02-01      0     0        0       0            

In [7]:
# --- time-series chart ---
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts.index, y=ts["total"], name="total quantum", line=dict(width=3, color="#1f77b4")))
for name, col in [("ionq","#2ca02c"),("rigetti","#d62728"),("d-wave","#9467bd"),("quantum computing","#ff7f0e")]:
    fig.add_trace(go.Scatter(x=ts.index, y=ts[name], name=name, line=dict(width=1.3, color=col)))
for label, d in MILESTONES:
    x = pd.Timestamp(d)
    fig.add_shape(type="line", x0=x, x1=x, y0=0, y1=1, yref="paper", line=dict(dash="dash", color="gray", width=1))
    fig.add_annotation(x=x, y=1.0, yref="paper", text=label, textangle=-35, showarrow=False, font=dict(size=8), yshift=8)
fig.update_layout(title=f"Quantum keyword headlines / month (2019–2023) · WQTM ETF benchmark {WQTM.date()}",
                  xaxis_title="month", yaxis_title="distinct headlines", template="plotly_white", height=460,
                  legend=dict(orientation="h", y=-0.2))
out = OUTPUT_DIR/"quantum_keyword_timeline.html"; fig.write_html(out)
print("saved chart ->", out.name)
fig.show()

saved chart -> quantum_keyword_timeline.html


In [8]:
print("first-seen per key term (full 2019-2023 window):")
for name, pat in KEY.items():
    m = df[df.Headline.str.contains(pat, case=False, na=False, regex=True)]
    print(f"  {name:18} first {m.date.min().date() if len(m) else '(none)'}  ·  {len(m):4} headlines")
print(f"\nall {len(hits):,} quantum headlines fall {(WQTM-hits.date.max()).days//30} months+ before WQTM ({WQTM.date()})")
print("\nsample distinct headlines around the IonQ SPAC (Feb-Mar 2021):")
for _, r in hits[(hits.date>='2021-02-20')&(hits.date<='2021-03-15')].sort_values("date").head(12).iterrows():
    print(f"  {r.date.date()}  {r.Headline[:92]}")

first-seen per key term (full 2019-2023 window):
  ionq               first 2021-02-24  ·   148 headlines
  rigetti            first 2017-03-28  ·    42 headlines
  d-wave             first 2013-05-16  ·    66 headlines
  quantum computing  first 2013-05-16  ·   139 headlines
  qubit              first 2014-09-29  ·     8 headlines
  quantinuum         first (none)  ·     0 headlines

all 385 quantum headlines fall 21 months+ before WQTM (2025-10-09)

sample distinct headlines around the IonQ SPAC (Feb-Mar 2021):
  2021-02-24  *IONQ SAID IN TALKS TO GO PUBLIC THROUGH MERGER WITH DMY SPAC
  2021-02-24  IonQ Said in Talks to Go Public Through Merger with DMY SPAC
  2021-02-24  IonQ Said in Talks to Go Public Through Merger with DMY SPAC (1)
  2021-02-24  IonQ Said in Talks to Go Public Through Merger with DMY SPAC (2)
  2021-03-02  *SUMITOMO INVESTS IN QUANTUM COMPUTING SOFTWARE START-UP CLASSIQ
  2021-03-03  Israel Allocates $60 Million to Build First Quantum Computer
  2021-03-08  *ION